# Affine vertex algebras

The universal affine vertex algebra $V^k(\mathfrak g)$ of a Lie algebra $\mathfrak g$ with an invariant form is generated by one current for each basis vector of $\mathfrak g$, with OPEs given by the bracket and the level $k$ times the form [Kac1998]. This notebook shows how to construct and compute with $V^k(\mathfrak g)$, for Lie algebras and Lie superalgebras $\mathfrak g$:

| construction | what it gives | section |
|---|---|---|
| `va.Affine(R, g, k)` | $V^k(\mathfrak g)$ for a Cartan type, a `LieSuperalgebra`, a Lie algebra of Sage, or structure constants | 1, 2 |
| `UniversalVertexAlgebra(R, X, opes)` | the same algebra written out by hand | 3 |
| `conformal_vector='sugawara'` | the Sugawara vector, away from the critical level | 4 |
| `x.horizontal_grade()`, `V.weight_space(n, horizontal_grade=a)` | the grading by the root lattice | 5 |
| `V.find_singular(n, ...)`, `V / I` | singular vectors and simple quotients $L_k(\mathfrak{sl}_2)$ at levels 1 and 2 | 6 |

**Prerequisites:** `01_getting_started.ipynb`, `02_lie_superalgebras.ipynb`.

**Elsewhere:** the Zhu algebras of affine vertex algebras and of their quotients are in `09_zhu_algebras.ipynb`, sections 2 and 6; their modules (Weyl modules, simple modules, affine Verma modules at admissible levels) in `10_modules.ipynb`; W-algebras, as quantum Hamiltonian reductions of affine vertex algebras, in `08_w_algebras.ipynb`; the research example on singular vectors of $V^k(\mathfrak g)$ at non-integral levels in `13_singular_vectors_in_affine_vertex_algebras.ipynb`.

**Contents**

1. The OPEs, the invariant form and the level
2. Four ways to give the Lie algebra
3. Building it by hand
4. The Sugawara vector and the critical level
5. The root lattice grading
6. Singular vectors and the simple quotient at integrable level
7. Limitations

## 1. The OPEs, the invariant form and the level

For a Lie (super)algebra $\mathfrak g$ with an invariant supersymmetric form $(\cdot|\cdot)$, the universal affine vertex algebra $V^k(\mathfrak g)$ is strongly generated by currents $J^a$, one for each basis vector $a$ of $\mathfrak g$, with

$$J^a(z)J^b(w) \sim \frac{k\,(a|b)}{(z-w)^2} + \frac{J^{[a,b]}(w)}{z-w}$$

[Kac1998, FZ1992]. For a simple Lie algebra, the form is the Killing form divided by $2h^\vee$, so that the highest root $\theta$ has $(\theta|\theta) = 2$. `va.Affine(R, g, k)` takes the coefficient ring, the Lie algebra and the level. Given a Cartan type, the basis is Sage's Chevalley basis: `alpha[1]` is $e_{\alpha_1}$, `alphacheck[1]` is $h_1$, and `-alpha[1]` is $f_{\alpha_1}$. In the cell below, the level is a parameter $k$ in $\mathbb Q(k)$, and the cell prints the generators of $V^k(\mathfrak{sl}_2)$ and three OPEs, as lists of poles: the first-order pole first.

In [1]:
import vertex_algebras as va
from vertex_algebras import *

K.<k> = Frac(QQ['k'])
V = va.Affine(K, ['A', 1], k)
e, h, f = V.gens()
V.variable_names(), e.ope(f), h.ope(e), h.ope(h)

(('alpha[1]', 'alphacheck[1]', '-alpha[1]'),
 [alphacheck[1], k],
 [2*alpha[1]],
 [0, 2*k])

So $e(z)f(w) \sim k/(z-w)^2 + h(w)/(z-w)$, $h(z)e(w) \sim 2e(w)/(z-w)$ and $h(z)h(w) \sim 2k/(z-w)^2$. In the cell below, `invariant_form_matrix()` is the Gram matrix of $k(\cdot|\cdot)$ in the order of the generators, that is, the matrix of the second-order poles. `killing_form_matrix()` is the Killing form $\operatorname{tr}(\operatorname{ad}a \operatorname{ad}b)$, which for $\mathfrak{sl}_2$ is $2h^\vee = 4$ times the normalised form.

In [2]:
V.invariant_form_matrix(), V.killing_form_matrix()

(
[  0   0   k]  [0 0 4]
[  0 2*k   0]  [0 8 0]
[  k   0   0], [4 0 0]
)

For $\mathfrak{sl}_3$, the generators are the six root vectors and two Cartan elements. The form pairs $e_\alpha$ with $e_{-\alpha}$, and on the Cartan subalgebra it is $k$ times the Cartan matrix. The level can also be a number: the last line of the cell below works at $k = \frac52$ over $\mathbb Q$.

In [3]:
V3 = va.Affine(K, ['A', 2], k)
print(V3.variable_names())
print(V3.invariant_form_matrix())
W = va.Affine(QQ, ['A', 2], 5/2)
W.gen('alpha[1]').ope(W.gen('-alpha[1]'))

('alpha[2]', 'alpha[1]', 'alpha[1] + alpha[2]', 'alphacheck[1]', 'alphacheck[2]', '-alpha[2]', '-alpha[1]', '-alpha[1] - alpha[2]')
[  0   0   0   0   0   k   0   0]
[  0   0   0   0   0   0   k   0]
[  0   0   0   0   0   0   0   k]
[  0   0   0 2*k  -k   0   0   0]
[  0   0   0  -k 2*k   0   0   0]
[  k   0   0   0   0   0   0   0]
[  0   k   0   0   0   0   0   0]
[  0   0   k   0   0   0   0   0]


[alphacheck[1], 5/2]

For a Lie superalgebra (see `02_lie_superalgebras.ipynb` and [Kac1977]), the currents of the odd basis vectors are odd, and the form is normalised as in the tables of [KW2004]. In $\mathfrak{sl}(2|1)$ the simple root $\alpha_2$ is odd and isotropic. The cell below shows that $x = $ `E[alpha[2]]` and $y = $ `E[-alpha[2]]` are odd, that $x(z)x(w) \sim 0$, and that the second-order pole changes sign when the two odd currents are exchanged, as the supersymmetry of the form requires.

In [4]:
S = LieSuperalgebra(QQ, 'sl(2|1)')
VS = va.Affine(K, S, k)
x, y = VS.gen('E[alpha[2]]'), VS.gen('E[-alpha[2]]')
x.parity(), x.ope(y), y.ope(x), x.ope(x)

(1, [h2, k], [h2, -k], [])

In $\mathfrak{osp}(1|2)$ (`spo(2|1)`), the odd simple root is not isotropic: $[x, x] = e_{2\alpha}$, so $x(z)x(w) \sim e_{2\alpha}(w)/(z-w)$, as the cell below shows.

In [5]:
O = va.Affine(K, LieSuperalgebra(QQ, 'spo(2|1)'), k)
xo = O.gen('E[alpha[1]]')
O.variable_names(), xo.ope(xo), xo.ope(O.gen('E[-alpha[1]]'))

(('E[alpha[1]]', 'E[2*alpha[1]]', 'h1', 'E[-alpha[1]]', 'E[-2*alpha[1]]'),
 [E[2*alpha[1]]],
 [h1, 4*k])

## 2. Four ways to give the Lie algebra

`va.Affine(R, g, k, form=None, ...)` accepts four kinds of input `g`:
- a Cartan type or Cartan matrix of a simple Lie algebra (section 1);
- a `LieSuperalgebra`, also one without odd roots;
- a Lie algebra of Sage with a finite basis, together with the invariant form `form` (a function of two elements, or a Gram matrix);
- structure constants `{(a, b): {c: coefficient}}`, with `parities={a: 1}` for the odd basis vectors, and `form` a dict, a matrix or a function.

For the last two, the level multiplies `form`. The constructor checks the (super) Jacobi identity of $\mathfrak g$ and the invariance of the form, which are exactly the Jacobi identities of the vertex algebra. The cell below gives $\mathfrak{sl}_2$ in all four ways, and prints the generators and two OPEs of the last three. For Sage's Lie algebra, the normalised form is the Killing form divided by $2h^\vee = 4$.

In [6]:
g = LieAlgebra(QQ, cartan_type=['A', 1])
sl2 = {('h', 'e'): {'e': 2}, ('h', 'f'): {'f': -2}, ('e', 'f'): {'h': 1}}
V_cartan = va.Affine(K, ['A', 1], k)
V_super = va.Affine(K, LieSuperalgebra(QQ, CartanMatrix(['A', 1])), k)
V_sage = va.Affine(K, g, k, form=lambda a, b: a.killing_form(b) / 4)
V_struct = va.Affine(K, sl2, k, form={('e', 'f'): 1, ('h', 'h'): 2}, names=['e', 'h', 'f'])
for A in (V_super, V_sage, V_struct):
    print(A.variable_names(), A.gen(0).ope(A.gen(2)), A.gen(1).ope(A.gen(0)))

('E[alpha[1]]', 'h1', 'E[-alpha[1]]') [h1, k] [2*E[alpha[1]]]
('alpha[1]', 'alphacheck[1]', '-alpha[1]') [alphacheck[1], k] [2*alpha[1]]
('e', 'h', 'f') [h, k] [2*e]


The four constructions give four different objects, since the data differ, at least in the names of the generators. They are isomorphic vertex algebras: in each, the generators come in the order $e, h, f$, and `V.hom(images)` checks that sending the generators of `V` to `images` respects all the OPEs, and raises an error otherwise (`10_modules.ipynb`, section 9). So each call in the cell below succeeds, and gives an isomorphism, since it maps the generators bijectively to the generators.

In [7]:
[V_cartan.hom(A.gens()).codomain() is A for A in (V_super, V_sage, V_struct)]

[True, True, True]

A Lie superalgebra can also be given by structure constants. `structure_coefficients()` of $\mathfrak{sl}(2|1)$ is in the format the constructor takes, and `invariant_form_matrix()` gives the form in the order of the basis; the parities are read off the basis. The cell below builds the algebra, and shows that it is the algebra `VS` of section 1 up to the grading, which only the `LieSuperalgebra` input provides.

In [8]:
names = [str(b) for b in S.basis().keys()]
odd = {str(b): 1 for b in S.basis().keys() if S.basis()[b].is_even_odd()}
VS_struct = va.Affine(K, S.structure_coefficients(), k, form=S.invariant_form_matrix(),
                      parities=odd, names=names)
print(VS_struct)
VS.hom(VS_struct.gens()).codomain() is VS_struct, VS_struct.horizontal_grading(), VS.horizontal_grading()

Universal affine vertex superalgebra of a Lie superalgebra of dimension 8 over Fraction Field of Univariate Polynomial Ring in k over Rational Field


(True,
 None,
 {'E[alpha[1]]': (1, 0),
  'E[alpha[2]]': (0, 1),
  'E[alpha[1] + alpha[2]]': (1, 1),
  'h1': (0, 0),
  'h2': (0, 0),
  'E[-alpha[1]]': (-1, 0),
  'E[-alpha[2]]': (0, -1),
  'E[-alpha[1] - alpha[2]]': (-1, -1)})

Wrong data are refused. In the cell below, the bracket $[e, f]$ is doubled, so the form is no longer invariant, and the constructor raises a `ValueError`:

In [9]:
bad = dict(sl2); bad[('e', 'f')] = {'h': 2}
va.Affine(K, bad, k, form={('e', 'f'): 1, ('h', 'h'): 2})

ValueError: the form is not invariant: ([e, f], h) = 4*k but (e, [f, h]) = 2*k

## 3. Building it by hand

The OPEs of section 1 can also be written out directly as a `UniversalVertexAlgebra` (`01_getting_started.ipynb`, section 2): one generator for each Chevalley basis vector of Sage's $\mathfrak{sl}_3$, with the bracket as the first-order pole and $k$ times the normalised Killing form as the second-order pole. The normalisation factor is $s = 1/(e_\theta|e_{-\theta})_{\text{Killing}}$, which is $1/(2h^\vee) = \frac16$. The OPE coefficients are elements of `NormallyOrderedExpressions`, and each pair of generators is given once. The cell below builds the algebra and prints $s$.

In [10]:
g3 = LieAlgebra(QQ, cartan_type=['A', 2])
B = g3.basis(); keys = list(B.keys()); labels = [str(a) for a in keys]
theta = g3.cartan_type().root_system().root_lattice().highest_root()
s = 1 / B[theta].killing_form(B[-theta])

X = NormallyOrderedExpressions(K, labels)
def J(v):      # the current of a Lie algebra element
    return X.sum(c * X.gen(str(a)) for a, c in v.monomial_coefficients().items())

opes = {(labels[i], labels[j]): [J(B[a].bracket(B[b])), k * s * B[a].killing_form(B[b])]
        for i, a in enumerate(keys) for j, b in enumerate(keys) if i <= j}
Vhand = UniversalVertexAlgebra(K, X, opes, degrees={a: 1 for a in labels})
s, Vhand

(1/6,
 Universal vertex algebra on generators ['alpha[2]', 'alpha[1]', 'alpha[1] + alpha[2]', 'alphacheck[1]', 'alphacheck[2]', '-alpha[2]', '-alpha[1]', '-alpha[1] - alpha[2]'] over Fraction Field of Univariate Polynomial Ring in k over Rational Field)

The generators come in the same order as in `V3 = va.Affine(K, ['A', 2], k)` of section 1, so the identity on generators is a morphism, and the weight spaces have the same dimensions, as the cell below checks.

In [11]:
(V3.hom(Vhand.gens()).codomain() is Vhand,
 [Vhand.weight_space(n).dimension() for n in range(4)],
 [V3.weight_space(n).dimension() for n in range(4)])

(True, [1, 8, 44, 192], [1, 8, 44, 192])

## 4. The Sugawara vector and the critical level

For a simple Lie algebra and $k \ne -h^\vee$, the Sugawara vector

$$T = \frac{1}{2(k + h^\vee)} \sum_a {:}J^a J_a{:},$$

where $J_a$ are the currents of the dual basis for $(\cdot|\cdot)$, is a conformal vector of central charge $c = k\dim\mathfrak g/(k + h^\vee)$, for which all the currents are primary of conformal weight 1 [Kac1998]. It is a choice, made with `conformal_vector='sugawara'`. The cell below takes $\mathfrak{sl}_2$, with the generators named $E, H, F$ by `names=`, and checks that $T = \frac{1}{2(k+2)}\big({:}EF{:} + {:}FE{:} + \frac12{:}HH{:}\big)$; the printed form uses ${:}FE{:} = {:}EF{:} - \partial H$.

In [12]:
Vs = va.Affine(K, ['A', 1], k, names=['E', 'H', 'F'], conformal_vector='sugawara')
E, H, F = Vs.gens()
T = Vs.conformal_vector()
T, T == (E.no(F) + F.no(E) + H.no(H) / 2) / (2 * (k + 2))

((1/(k + 2))*:E F: + (1/4/(k + 2))*:H H: + (-1/2/(k + 2))*dH, True)

The cell below prints the central charge $3k/(k+2)$ of $V^k(\mathfrak{sl}_2)$, the conformal weights of the generators, and the OPE $T(z)E(w) \sim E(w)/(z-w)^2 + \partial E(w)/(z-w)$, which says that $E$ is primary of weight 1. For $\mathfrak{sl}_3$, $c = 8k/(k+3)$.

In [13]:
(Vs.central_charge(), Vs.degrees(), T.ope(E),
 va.Affine(K, ['A', 2], k, conformal_vector='sugawara').central_charge())

(3*k/(k + 2), {'E': 1, 'H': 1, 'F': 1}, [dE, E], 8*k/(k + 3))

For a Lie superalgebra, the library uses $T = \tfrac12\sum_{a,b}\Omega_{ab}{:}J_aJ_b{:}$ with $\Omega = (B + \kappa/2)^{-1}$, where $B$ is the level times the form and $\kappa$ is the super Killing form. For simple $\mathfrak g$, $\kappa = 2h^\vee(\cdot|\cdot)$, so $B + \kappa/2 = (k + h^\vee)(\cdot|\cdot)$ and this is the formula above. The central charge is $c = k\operatorname{sdim}\mathfrak g/(k + h^\vee)$ [KRW2003]. For $\mathfrak{sl}(2|1)$, $\operatorname{sdim} = 0$, so $c = 0$; for $\mathfrak{osp}(1|2)$, $\operatorname{sdim} = 1$ and $h^\vee = \frac32$. The cell below prints, for both, the central charge, the superdimension and $h^\vee$.

In [14]:
[(G, va.Affine(K, G, k, conformal_vector='sugawara').central_charge(), G.superdimension(), G.dual_coxeter_number())
 for G in (LieSuperalgebra(QQ, 'sl(2|1)'), LieSuperalgebra(QQ, 'spo(2|1)'))]

[(Lie superalgebra sl(2|1) over Rational Field, 0, 0, 1),
 (Lie superalgebra spo(2|1) over Rational Field, k/(k + 3/2), 1, 3/2)]

A rational level works in the same way: for $\mathfrak{sl}_2$ at $k = \frac12$, $c = \frac{3/2}{5/2} = \frac35$.

In [15]:
va.Affine(QQ, ['A', 1], 1/2, conformal_vector='sugawara').central_charge()

3/5

**The critical level.** At $k = -h^\vee$, the factor $1/(k + h^\vee)$ does not exist. The library raises a `ValueError` whenever $B + \kappa/2$ is not invertible over the coefficient ring: at the critical level, as in the cell below, and also over a polynomial ring such as $\mathbb Q[k]$, where $k + h^\vee$ has no inverse.

In [16]:
va.Affine(QQ, ['A', 1], -2, conformal_vector='sugawara')

ValueError: no Sugawara vector: B + kappa/2 is not invertible over Rational Field (the critical level?)

At the critical level, the unnormalised vector $S = {:}EF{:} + {:}FE{:} + \frac12{:}HH{:}$ has no singular part in its OPE with any current, so $S$ lies in the centre of $V^{-2}(\mathfrak{sl}_2)$. At $k = 1$, instead, $J(z)S(w) \sim 6J(w)/(z-w)^2$ for every current $J$: there $S = 2(k + h^\vee)T = 6T$, and $J(z)T(w) \sim J(w)/(z-w)^2$. The cell below prints the OPEs of the currents with $S$ at both levels.

In [17]:
for lev in (-2, 1):
    A = va.Affine(QQ, ['A', 1], lev, names=['E', 'H', 'F'])
    a, b, c = A.gens()
    S2 = a.no(c) + c.no(a) + b.no(b) / 2
    print(lev, [x.ope(S2) for x in A.gens()])

-2 [[], [], []]
1 [[0, 6*E], [0, 6*H], [0, 6*F]]


## 5. The root lattice grading

Given a Cartan type or a `LieSuperalgebra`, the algebra is graded by the root lattice: the current of a root vector has the root as its grade, written in simple root coordinates, and the Cartan currents have grade 0. Grades add under all products. `x.horizontal_grade()` is the grade of a homogeneous element, and `V.horizontal_grading_group()` is the group of grades, $\mathbb Z^r$ for a Lie algebra of rank $r$. The cell below prints the grades of $E$, $H$, $F$ and ${:}EE{:}$, and the grading group.

In [18]:
E.horizontal_grade(), H.horizontal_grade(), F.horizontal_grade(), E.no(E).horizontal_grade(), Vs.horizontal_grading_group()

((1),
 (0),
 (-1),
 (2),
 Ambient free module of rank 1 over the principal ideal domain Integer Ring)

For $\mathfrak{sl}_2$ the grading group is $\mathbb Z^1$, not $\mathbb Z$, and Sage differentiates these objects: in every rank a grade is a vector of simple root coordinates, so $E$ has grade $(1)$ just as, for $\mathfrak{sl}_3$ below, $\theta$ has grade $(1, 1)$. Sage prints a vector with one entry without a comma, as `(1)`. In Python, however, `(1)` is the integer 1, so a grade passed as an argument is written as the tuple `(1,)`, as in the cells below. The lattice vertex algebra $V_{A_1}$ of section 6 is graded by $\mathbb Z$ instead: its lattice $\mathbb Z\alpha$ is the root lattice of $\mathfrak{sl}_2$, and the grade `(m,)` of $L_1(\mathfrak{sl}_2)$ is the grade `m` of $V_{A_1}$.

`V.weight_space(n, horizontal_grade=a)` is the part of conformal weight $n$ and grade $a$. The first cell below prints, for $\mathfrak{sl}_2$ at the weights $n = 0, \dots, 3$, the dimensions of the grades $-n, \dots, n$; the second prints the bases of weight 2 in the grades 2 and 1.

In [19]:
[[Vs.weight_space(n, horizontal_grade=(a,)).dimension() for a in range(-n, n + 1)] for n in range(4)]

[[1], [1, 1, 1], [1, 2, 3, 2, 1], [1, 2, 5, 6, 5, 2, 1]]

In [20]:
Vs.weight_space(2, horizontal_grade=(2,)).basis(), Vs.weight_space(2, horizontal_grade=(1,)).basis()

([:E E:], [:E H:, dE])

`V.character(N)` is $\sum_{n<N}\dim V_n\,q^n$, and with `horizontal_grade=` it is the character of one grade. As a vector space, $V^k(\mathfrak g)$ is the symmetric algebra on the negative modes of the currents, so its character is $\prod_{n\ge1}(1-q^n)^{-\dim\mathfrak g}$, here $\prod_{n \ge 1}(1-q^n)^{-3}$. The cell below checks this up to $q^6$ and prints the character of grade 0.

In [21]:
ch = Vs.character(7)
P.<q> = PowerSeriesRing(QQ, default_prec=7)
ch, ch.laurent_part() == (1 / prod((1 - q^n)^3 for n in range(1, 7))).truncate(7), Vs.character(7, horizontal_grade=(0,))

(1 + 3*q + 9*q^2 + 22*q^3 + 51*q^4 + 108*q^5 + 221*q^6 + O(q^7),
 True,
 1 + q + 3*q^2 + 6*q^3 + 13*q^4 + 24*q^5 + 47*q^6 + O(q^7))

For $\mathfrak{sl}_3$ the grades have two coordinates. In the cell below the generators are named $e_i, h_i, f_i$, because the default names such as `alpha[1] + alpha[2]` contain spaces and signs, so that, for example, $\partial$ of `alpha[1] + alpha[2]` prints as `dalpha[1] + alpha[2]`. The highest root $\theta = \alpha_1 + \alpha_2$ has grade $(1, 1)$, and at weight 2 this grade has four states. The grade $(0,0)$ at weight 2 has dimension 8: three products ${:}e_\alpha f_\alpha{:}$, three products ${:}h_ih_j{:}$ and two derivatives $\partial h_i$.

In [22]:
V3s = va.Affine(K, ['A', 2], k, names=['e2', 'e1', 'e12', 'h1', 'h2', 'f2', 'f1', 'f12'],
                conformal_vector='sugawara')
print([(x, x.horizontal_grade()) for x in V3s.gens()])
V3s.weight_space(2, horizontal_grade=(1, 1)).basis(), V3s.weight_space(2, horizontal_grade=(0, 0)).dimension()

[(e2, (0, 1)), (e1, (1, 0)), (e12, (1, 1)), (h1, (0, 0)), (h2, (0, 0)), (f2, (0, -1)), (f1, (-1, 0)), (f12, (-1, -1))]


([:e2 e1:, :e12 h1:, :e12 h2:, de12], 8)

## 6. Singular vectors and the simple quotient at integrable level

For a non-negative integer $k$, the simple quotient $L_k(\mathfrak g)$ of $V^k(\mathfrak g)$ is the quotient by the ideal generated by the singular vector $e_\theta(-1)^{k+1}\mathbf 1$ [Kac1990, FZ1992]. For $\mathfrak{sl}_2$ this vector is ${:}E^{k+1}{:}$.

`V.find_singular(n, horizontal_grade=a, zero_modes=[...])` computes a basis of the singular vectors of weight $n$ and grade $a$: the vectors killed by the positive modes of the generators and by the zero modes of the listed elements (here $E$, so that the singular vectors are highest-weight vectors for $\mathfrak{sl}_2$). The cell below shows that at $k = 1$ there is one of weight 2, ${:}EE{:}$, and none of weight 1.

In [23]:
V1 = va.Affine(QQ, ['A', 1], 1, names=['E', 'H', 'F'], conformal_vector='sugawara')
E1, H1, F1 = V1.gens()
(V1.find_singular(2, horizontal_grade=(2,), zero_modes=[E1]),
 [V1.find_singular(1, horizontal_grade=(a,), zero_modes=[E1]) for a in (-1, 0, 1)])

([:E E:], [[], [], []])

`V.ideal([v])` is the ideal generated by $v$, and `V.quotient(I)` (also `V / I`) is the quotient (`06_subalgebras_ideals_quotients.ipynb`, sections 3 and 4). A quotient has no conformal vector or grading unless asked for: `conformal_vector='ambient'` takes the image of the Sugawara vector, and `horizontal_grading='ambient'` the root lattice grading. The cell below prints the dimensions of the weight spaces of the ideal, which at weight 2 is the five-dimensional $\mathfrak{sl}_2$-module generated by ${:}EE{:}$, the central charge of the quotient, and shows that ${:}EE{:} = 0$ in $L_1(\mathfrak{sl}_2)$.

In [24]:
I = V1.ideal([E1.no(E1)])
L1 = V1.quotient(I, horizontal_grading='ambient', conformal_vector='ambient')
Eq, Hq, Fq = L1.gens()
[I.weight_space(n).dimension() for n in range(5)], L1.central_charge(), Eq.no(Eq), Eq.ope(Fq)

([0, 0, 5, 15, 38], 1, 0, [H, 1])

**An independent check.** $L_1(\mathfrak{sl}_2)$ is the lattice vertex algebra $V_{A_1}$ of the lattice $\mathbb Z\alpha$ with $(\alpha|\alpha) = 2$ [FK1980] (see `05_lattice_vertex_algebras.ipynb`), with $E \mapsto e^{\alpha}$, $H \mapsto J$ (the Heisenberg current) and $F \mapsto e^{-\alpha}$. The conformal vector of $V_{A_1}$ is $\frac14{:}JJ{:}$, and its character is $\sum_{m\in\mathbb Z} q^{m^2}/\prod_{n\ge1}(1-q^n)$. The cell below checks that the characters agree, also grade by grade (the lattice grade $m$ is the root grade $m$), and `hom` confirms that the map above is a morphism $L_1(\mathfrak{sl}_2) \to V_{A_1}$.

In [25]:
LA = va.Lattice(QQ, matrix([[2]])); Jl = LA.J(0)
LA = LA.with_conformal_vector(Jl.no(Jl) / 4).with_horizontal_grading('lattice')
print(L1.character(9))
print(LA.character(9))
print(L1.character(9) == LA.character(9),
      all(L1.character(9, horizontal_grade=(m,)) == LA.character(9, horizontal_grade=m) for m in range(-3, 4)))
L1.hom([LA.e((1,)), Jl, LA.e((-1,))])

1 + 3*q + 4*q^2 + 7*q^3 + 13*q^4 + 19*q^5 + 29*q^6 + 43*q^7 + 62*q^8 + O(q^9)
1 + 3*q + 4*q^2 + 7*q^3 + 13*q^4 + 19*q^5 + 29*q^6 + 43*q^7 + 62*q^8 + O(q^9)
True True


Vertex algebra morphism:
  From: Quotient of Universal affine vertex algebra of type ['A', 1] at level 1 over Rational Field by the ideal generated by [:E E:]
  To:   Lattice vertex algebra of rank 1 with Gram matrix [2] over Rational Field
  Defn: E |--> e^(1)
        H |--> J0
        F |--> e^(-1)

The grading group of the lattice vertex algebra is $\mathbb Z$, so its grades are integers, while that of the affine vertex algebra is $\mathbb Z^1$, with grades written `(m,)`.

At $k = 2$, the singular vector is ${:}EEE{:}$ (`V.no(E, E, E)`), of weight 3. Three free fermions $\psi_1, \psi_2, \psi_3$ give an independent check: their bilinears ${:}\psi_i\psi_j{:}$ generate a copy of $L_2(\mathfrak{sl}_2) \cong L_1(\mathfrak{so}_3)$, which is the integer-weight part of the free fermion vertex superalgebra [FF1985]. So the character of $L_2(\mathfrak{sl}_2)$ is the integer-weight part of $\prod_{n\ge1}(1+q^{n-1/2})^3$. The cell below finds the singular vector, builds the quotient and compares the two characters; in the series, $x = q^{1/2}$.

In [26]:
V2 = va.Affine(QQ, ['A', 1], 2, names=['E', 'H', 'F'], conformal_vector='sugawara')
E2 = V2.gen('E')
sing = V2.find_singular(3, horizontal_grade=(3,), zero_modes=[E2])
L2 = V2.quotient(sing, conformal_vector='ambient')
Px.<x> = PowerSeriesRing(QQ, default_prec=20)
fermions = prod((1 + x^(2*n - 1))^3 for n in range(1, 10))
sing, L2.central_charge(), L2.character(9), [fermions[2*n] for n in range(9)]

([:E E E:],
 3/2,
 1 + 3*q + 9*q^2 + 15*q^3 + 30*q^4 + 54*q^5 + 94*q^6 + 153*q^7 + 252*q^8 + O(q^9),
 [1, 3, 9, 15, 30, 54, 94, 153, 252])

The Zhu algebras of these quotients, and with them the classification of their modules, are in `09_zhu_algebras.ipynb`, section 6. The research example `13_singular_vectors_in_affine_vertex_algebras.ipynb` searches for singular vectors of $V^k(D_4)$ at the non-integrable level $k = -2$.

## 7. Limitations

* **Super Cartan types.** Sage's super Cartan types (such as `['A', [1, 0]]`) are refused with a `NotImplementedError`; give a `LieSuperalgebra` or structure constants instead.
* **The Sugawara vector** exists only when $B + \kappa/2$ is invertible over the coefficient ring: not at the critical level, and not over a polynomial ring in $k$ (use its fraction field instead).
* **Gradings.** Only Cartan types and `LieSuperalgebra`s come with the root lattice grading; for a Lie algebra of Sage or structure constants, pass `horizontal_grading=` explicitly.
* **Simple quotients.** The library does not compute the maximal ideal of $V^k(\mathfrak g)$: the generators of the ideal must be supplied, from theory as in section 6 or from `find_singular`. `find_singular` finds the singular vectors of one weight and one grade at a time, at a fixed level; the levels at which singular vectors exist are found in `13_singular_vectors_in_affine_vertex_algebras.ipynb`. At non-integrable levels the maximal ideal can need more generators, and the computation cannot show that the generators found suffice.
* **Cost.** The weight spaces grow like the coefficients of $\prod_{n \ge 1}(1-q^n)^{-\dim\mathfrak g}$, so large Lie algebras are only practical in low weights, or one grade at a time.

## References

- [Kac1998] V. G. Kac, *Vertex algebras for beginners*, 2nd ed., Univ. Lecture Ser. 10, AMS (1998), [doi:10.1090/ulect/010](https://doi.org/10.1090/ulect/010). — affine vertex algebras and the Sugawara construction (sections 1, 4).
- [FZ1992] I. B. Frenkel, Y. Zhu, *Vertex operator algebras associated to representations of affine and Virasoro algebras*, Duke Math. J. 66 (1992), 123–168, [doi:10.1215/S0012-7094-92-06604-X](https://doi.org/10.1215/S0012-7094-92-06604-X). — affine vertex operator algebras and their simple quotients at positive integer level (sections 1, 6).
- [Kac1977] V. G. Kac, *Lie superalgebras*, Adv. Math. 26 (1977), 8–96, [doi:10.1016/0001-8708(77)90017-2](https://doi.org/10.1016/0001-8708(77)90017-2). — basic classical Lie superalgebras (section 1).
- [KW2004] V. G. Kac, M. Wakimoto, *Quantum reduction and representation theory of superconformal algebras*, Adv. Math. 185 (2004), 400–458, [arXiv:math-ph/0304011](https://arxiv.org/abs/math-ph/0304011), [doi:10.1016/j.aim.2003.12.005](https://doi.org/10.1016/j.aim.2003.12.005); corrigendum, Adv. Math. 193 (2005), 453–455, [doi:10.1016/j.aim.2005.01.001](https://doi.org/10.1016/j.aim.2005.01.001). — the normalisation of the invariant forms of Lie superalgebras (section 1).
- [KRW2003] V. G. Kac, S.-S. Roan, M. Wakimoto, *Quantum reduction for affine superalgebras*, Comm. Math. Phys. 241 (2003), 307–342, [arXiv:math-ph/0302015](https://arxiv.org/abs/math-ph/0302015), [doi:10.1007/s00220-003-0926-1](https://doi.org/10.1007/s00220-003-0926-1). — the Sugawara vector of affine vertex superalgebras and its central charge (section 4).
- [Kac1990] V. G. Kac, *Infinite dimensional Lie algebras*, 3rd ed., Cambridge University Press (1990), [doi:10.1017/CBO9780511626234](https://doi.org/10.1017/CBO9780511626234). — integrable highest-weight modules of affine Lie algebras (section 6).
- [FK1980] I. B. Frenkel, V. G. Kac, *Basic representations of affine Lie algebras and dual resonance models*, Invent. Math. 62 (1980), 23–66, [doi:10.1007/BF01391662](https://doi.org/10.1007/BF01391662). — $L_1(\mathfrak{sl}_2)$ as the $A_1$ lattice vertex algebra (section 6).
- [FF1985] A. J. Feingold, I. B. Frenkel, *Classical affine algebras*, Adv. Math. 56 (1985), 117–172. — fermionic constructions of the classical affine Lie algebras, here $L_1(\mathfrak{so}_3) \cong L_2(\mathfrak{sl}_2)$ from three free fermions (section 6).